In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
import json
curr_wd = os.path.abspath(os.getcwd())
print(curr_wd)
from importlib import reload


In [ ]:
import src.gather_genomic_coordinates as gather_genomic_coordinates
# Reload the module after making changes
reload(gather_genomic_coordinates)

################POSITIVE######################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "pos_RG_regions_win5.json"

regions = gather_genomic_coordinates._read_input(f"{path}/{filename}")
pos_genomic_coordinates_json, failed_pos = gather_genomic_coordinates.process_multiple_regions(regions, parallel=False)
print(pos_genomic_coordinates_json)
print(failed_pos)

##################NEGATIVE##########

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "neg_RG_regions_win5.json"

regions = gather_genomic_coordinates._read_input(f"{path}/{filename}")
neg_genomic_coordinates_json, failed_neg = gather_genomic_coordinates.process_multiple_regions(regions, parallel=False)
print(neg_genomic_coordinates_json)
print(failed_neg)


In [ ]:
### HIER MUSS ICH NOCHMAL SCHAUEN, WELCHE FAILEN UND WELCHE NEN TRANSLATION MISMATCH HABEN

In [ ]:
for i,el in enumerate(pos_genomic_coordinates_json):
    curr_dict = pos_genomic_coordinates_json[i]
    curr_dict["region_id"] = str(el['protein'] + '_' + str(el['prot_region'][0]) + '_' + str(el['prot_region'][1]))
    pos_genomic_coordinates_json[i] = curr_dict

for i,el in enumerate(neg_genomic_coordinates_json):
    curr_dict = neg_genomic_coordinates_json[i]
    curr_dict["region_id"] = str(el['protein'] + '_' + str(el['prot_region'][0]) + '_' + str(el['prot_region'][1]))
    neg_genomic_coordinates_json[i] = curr_dict

merged_genomic_coordinates_list = [
    {**d, "group": "pos"} for d in pos_genomic_coordinates_json
] + [
    {**d, "group": "neg"} for d in neg_genomic_coordinates_json
]


path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "genomic_coords_merged_win5.json"

with open(f"{path}/{filename}", "w") as fh:
    json.dump(merged_genomic_coordinates_list, fh, indent=4)

In [ ]:
###### TURN INTO BED FILE FOR MOGON ANALYSIS

# import src.write_bed_file as write_bed_file
# # Reload the module after making changes
# reload(write_bed_file)


# path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
# filename = "genomic_coords_pos_win5.bed"

# write_bed_file.write_bed_from_results(
#     pos_genomic_coordinates_json,
#     f"{path}/{filename}"
# )

# ###################################

# path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
# filename = "genomic_coords_neg_win5.bed"

# write_bed_file.write_bed_from_results(
#     neg_genomic_coordinates_json,
#     f"{path}/{filename}"
# )

### USE MERGED FILE!!

import src.write_bed_file as write_bed_file

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "genomic_coords_combined_win5.bed"

write_bed_file.write_merged_bed(
    pos_genomic_coordinates_json,
    neg_genomic_coordinates_json,
    f"{path}/{filename}"
)



In [ ]:
# NEW RUN IN MOGON, INCL VEP COLUMNS FORM THE VCF FILE
# Download the ready tsv from the Mogon

In [ ]:
import src.variant_assignment as va

# Load
df_raw = va.load_vep_tsv(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/bcftools/combined_vep_variants.tsv"
)

# Quick peek before filtering
# print(df_raw.head())
print(df_raw.columns.tolist())
print(df_raw["FILTER"].value_counts().head(5))
print(df_raw["BIOTYPE"].value_counts().head(5))
print(df_raw["Consequence"].value_counts().head(5))

# Filter
df_filt = va.filter_vep(df_raw)
# print(df_filt.head())

In [ ]:
# How complete is UniProt in the filtered set?
print("Rows with UNIPROT_ISOFORM populated:", df_filt["UNIPROT_ISOFORM"].notna().sum())
print("Rows without:", df_filt["UNIPROT_ISOFORM"].isna().sum())
print()
print("Sample of populated values:")
print(df_filt["UNIPROT_ISOFORM"].dropna().sample(10).tolist())
print()
print("Gene symbols where UNIPROT_ISOFORM is missing:")
print(df_filt[df_filt["UNIPROT_ISOFORM"].isna()]["SYMBOL"].value_counts().head(10))

In [ ]:
import json

with open("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/genomic_coords_merged_win5.json") as f:
    regions = json.load(f)

uniprot_ids = sorted({r["protein"] for r in regions})
print(f"Unique UniProt IDs in regions: {len(uniprot_ids)}")
print(uniprot_ids[:10])

In [ ]:
import src.variant_assignment as va

uniprot_to_symbols = va.fetch_uniprot_to_symbols(uniprot_ids)

# Check coverage
print(f"Got mappings for {len(uniprot_to_symbols)} / {len(uniprot_ids)} UniProt IDs")

# Any missing?
missing = set(uniprot_ids) - set(uniprot_to_symbols.keys())
if missing:
    print(f"Missing: {missing}")

# Sanity check — does RBMXL2 appear in any reverse mapping?
for acc, symbols in uniprot_to_symbols.items():
    if "RBMXL2" in symbols:
        print(f"RBMXL2 → UniProt {acc}")
        break

In [ ]:
"O75526" in {r["protein"] for r in regions}

In [ ]:
uniprot_lookup, symbol_lookup = va.build_region_lookups(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/genomic_coords_merged_win5.json",
    uniprot_to_symbols,
)

df_assigned = va.assign_variants_to_regions(df_filt, uniprot_lookup, symbol_lookup)
df_assigned.head()

In [ ]:
###### NOW ALL VARIANT ARE MAPPED TO MY REGIONS!
#### NOW WE ADD BACK THE ALLELE FREQUENCIES THAT THE VCF FROM 

In [ ]:
df_with_af = va.load_and_merge_frequencies(
    df_assigned,
    pos_tsv_path="/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/combined_joint_variants_pos_win5_extended.tsv",
    neg_tsv_path="/mnt/d/phd/scripts/14_gnomAD_conservation_proj/data/output/combined_joint_variants_neg_win5_extended.tsv",
)

# Quick audit
print("\nAF coverage by group:")
print(df_with_af.groupby("group")["AF_joint"].apply(
    lambda s: f"{s.notna().sum()} / {len(s)} ({s.notna().mean():.1%})"
))

print("\nAF distribution:")
print(df_with_af["AF_joint"].describe())

print("\nRow with missing AF (sample):")
print(df_with_af[df_with_af["AF_joint"].isna()].head(3)[
    ["CHROM","POS","REF","ALT","SYMBOL","region_id","group"]
])

In [ ]:
df_with_af = va.parse_amino_acids_column(df_with_af)

# Sanity check across consequence types
sample_cons = ["missense_variant", "synonymous_variant", "stop_gained",
               "inframe_deletion", "inframe_insertion", "start_lost"]
for cons in sample_cons:
    subset = df_with_af[df_with_af["Consequence"] == cons]
    if len(subset) > 0:
        print(f"\n{cons}:")
        print(subset[["Amino_acids", "before_aa", "after_aa"]].head(3).to_string(index=False))

print(f"\nNulls in before_aa: {df_with_af['before_aa'].isna().sum()}")
print(f"Nulls in after_aa: {df_with_af['after_aa'].isna().sum()}")

In [ ]:
# Extract the UniProt IDs from our regions
import json
with open("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/genomic_coords_merged_win5.json") as f:
    regions = json.load(f)
uniprot_ids = sorted({r["protein"] for r in regions})

# Load + filter AlphaMissense
am = va.load_alphamissense_for_proteins(
    am_tsv_path="/mnt/d/phd/scripts/16_ev_signature_predictor/data/alphamissense/AlphaMissense_aa_substitutions.tsv.gz",
    uniprot_ids=uniprot_ids,
)
am.head()

In [ ]:
am.to_parquet(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/alphamissense/am_filtered_for_my_proteins.parquet"
)
print("Saved.")

In [ ]:
df_final = va.merge_alphamissense(df_with_af, am)

# Missing protein check
missing = set(uniprot_ids) - set(am["uniprot_id"])
print(f"\nUniProt IDs missing from AlphaMissense: {missing}")

# Distribution check
missense = df_final[df_final["Consequence"].str.contains("missense_variant", na=False)]
print("\nPathogenicity by group (missense only):")
print(missense.groupby("group")["am_pathogenicity"].describe())

print("\nClass breakdown by group:")
print(missense.groupby(["group", "am_class"]).size().unstack(fill_value=0))

In [ ]:
print("pos genes:", df_final[df_final["group"]=="pos"]["SYMBOL"].nunique())
print("neg genes:", df_final[df_final["group"]=="neg"]["SYMBOL"].nunique())
print("shared:", len(set(df_final[df_final["group"]=="pos"]["SYMBOL"]) & 
                     set(df_final[df_final["group"]=="neg"]["SYMBOL"])))

In [ ]:
# Are pos proteins under higher gene-level constraint than neg proteins?
# Proxy: within each gene, what's the overall pathogenicity of all missense variants?
gene_level = df_final[
    df_final["Consequence"].str.contains("missense_variant", na=False)
].groupby(["group", "SYMBOL"])["am_pathogenicity"].median().reset_index()
print(gene_level.groupby("group")["am_pathogenicity"].describe())

In [ ]:
af_cols = [c for c in df_final.columns if c.startswith(("AC_", "AN_", "AF_"))]
print(f"Fixing {len(af_cols)} AF columns")

for col in af_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

# Verify
print(df_final[af_cols].dtypes.value_counts())

In [ ]:
df_final.to_parquet(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/variants_annotated_final.parquet"
)
print(f"Saved {len(df_final):,} rows × {len(df_final.columns)} columns")